# Getting Started

tsam_xarray wraps [tsam](https://github.com/FZJ-IEK3-VSA/tsam) for xarray DataArrays.
This notebook shows the basic workflow.

## Sample data

tsam_xarray includes sample energy data with realistic profiles for documentation.

In [ ]:
import xarray_plotly  # noqa: F401 — registers .plotly accessor

import tsam_xarray
from tsam_xarray._sample_data import sample_energy_data

da = sample_energy_data(n_days=30)
da

In [ ]:
da.sel(region="north", scenario="low").plotly.line(x="time", color="variable")

## Simple case: 2D DataArray

For a `(time, variable)` array, `column_dims` is auto-detected.

In [ ]:
# Pick one region and scenario for the simple case
da_simple = da.sel(region="north", scenario="low")

result = tsam_xarray.aggregate(da_simple, n_clusters=4)
result.typical_periods

In [ ]:
result.typical_periods.plotly.line(x="timestep", facet_col="variable", color="cluster")

## Inspect results

The result contains xarray-native fields.

In [ ]:
print(f"Clusters: {result.n_clusters}")
print(f"Timesteps per period: {result.n_timesteps_per_period}")
print("Cluster weights (days each represents):")
result.cluster_weights

In [ ]:
result.accuracy.rmse

## Reconstructed time series

The full time series rebuilt from typical periods — same shape as the input.

In [ ]:
result.reconstructed

In [ ]:
import xarray as xr

comparison = xr.concat(
    [
        da_simple.sel(variable="solar").rename("original"),
        result.reconstructed.sel(variable="solar").rename("reconstructed"),
    ],
    dim="source",
).assign_coords(source=["original", "reconstructed"])
comparison.plotly.line(x="time", color="source")

## Residuals

In [ ]:
result.residuals

## Passing tsam parameters

All `tsam.aggregate()` keyword arguments pass through.

In [ ]:
from tsam import ClusterConfig

result_km = tsam_xarray.aggregate(
    da_simple,
    n_clusters=4,
    cluster=ClusterConfig(method="kmeans"),
)
result_km.accuracy.rmse